# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [129]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [130]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [131]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [132]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [133]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [134]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [135]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [136]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [137]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [138]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [139]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18


### La soumission


In [140]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [141]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [142]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "…"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 149894 lignes sur 149894   (identifiant non renseigné : la table entière)


### L'interface, telle qu'on l'appelle


In [143]:
print(score_interne)

82.50775542880015


In [144]:
import glob
sub = pd.read_csv("submission.csv")
print("ma soumission :", sub.shape, list(sub.columns))
print(sub.head(3))
print("ids uniques :", sub[ID].is_unique, "| lignes du test :", len(test), "| ids du test uniques :", test[ID].is_unique)

fichiers = glob.glob(os.path.join(DOSSIER, "*"))
print("fichiers de la compétition :", [os.path.basename(f) for f in fichiers])

ex_path = [f for f in fichiers if "sample" in f.lower() or "submission" in f.lower()]
if ex_path:
    ex = pd.read_csv(ex_path[0])
    print("exemple :", ex.shape, list(ex.columns))
    print(ex.head(3))
    print("mêmes ids que l'exemple :", set(ex[ID]) == set(sub[ID]))
    print("ids de l'exemple absents de ma soumission :", len(set(ex[ID]) - set(sub[ID])))
    print("ids de ma soumission absents de l'exemple :", len(set(sub[ID]) - set(ex[ID])))

ma soumission : (63721, 2) ['ligne_id', 'patient_absent']
   ligne_id  patient_absent
0     44088               0
1    120507               0
2     46612               0
ids uniques : True | lignes du test : 63721 | ids du test uniques : True
fichiers de la compétition : ['sample_submission.csv', 'table_activite.csv', 'train.csv', 'test.csv']
exemple : (63721, 2) ['ligne_id', 'patient_absent']
   ligne_id  patient_absent
0     44088               0
1    120507               0
2     46612               0
mêmes ids que l'exemple : True
ids de l'exemple absents de ma soumission : 0
ids de ma soumission absents de l'exemple : 0


In [145]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [146]:
part_absents_controle = 100 * yval.mean()
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())

classe_majoritaire = 0 if yval.mean() < 0.5 else 1
pred = np.full(len(yval), classe_majoritaire)
print("score du prédicteur constant :", (pred == yval).mean() * 100)

ecart_a_train = ma_reference - reference

ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
print(part_absents_controle, ma_reference, ecart_a_train, ecart_collegue, erreur_type, ecart_en_barres)

score du prédicteur constant : 81.75055872444044
18.24944127555956 81.75055872444044 0.001456025199644273 1.2494412755595619 0.2194126110590912 5.694482507311615


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [147]:
def balayer(df, min_lignes=0):
    """Une ligne par colonne : écart max-min des taux d'absence (en points)
       entre ses groupes, en ne gardant que les groupes d'au moins `min_lignes` lignes."""
    res = {}
    for col in df.columns:
        if col == CIBLE:          # la cible ne se groupe pas par elle-même
            continue
        g = df.groupby(col)[CIBLE].agg(["mean", "size"])
        g = g[g["size"] >= min_lignes]
        if len(g) < 2:            # un seul groupe : rien à comparer
            continue
        res[col] = {"taux_max": 100 * g["mean"].max(),
                    "taux_min": 100 * g["mean"].min(),
                    "ecart": 100 * (g["mean"].max() - g["mean"].min()),
                    "plus_petit_groupe": int(g["size"].min()),
                    "groupes_de_500": int((g["size"] >= 500).sum())}
    return (pd.DataFrame.from_dict(res, orient="index")
              .sort_values("ecart", ascending=False))

# 1-2. balayage SANS filtre
balayage_brut = balayer(mon_train)
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut.iloc[0]["ecart"]
groupes_de_500 = int(balayage_brut.iloc[0]["groupes_de_500"])
display(balayage_brut.head(8))

# 3-4. balayage FILTRÉ : groupes d'au moins 500 lignes, puis comparaison
classement = balayer(mon_train, 500)
plus_petit_groupe_du_classement = int(classement["plus_petit_groupe"].min())
meilleure_colonne = classement.index[0]
ecart_filtre = classement.iloc[0]["ecart"]
display(classement.head(8))

,taux_max,taux_min,ecart,plus_petit_groupe,groupes_de_500
dossier_id,100.0,0.0,100.0,1,0
ligne_id,100.0,0.0,100.0,1,0
patient_id,100.0,0.0,100.0,1,0
taille_cm,100.0,0.0,100.0,1,140
indice_risque_externe,100.0,0.0,100.0,1,0
date_prise,100.0,0.0,100.0,1,0
cout_acte,100.0,0.0,100.0,1,119
montant_acte,100.0,0.0,100.0,1,0


,taux_max,taux_min,ecart,plus_petit_groupe,groupes_de_500
duree_consultation,99.400480,0.000000,99.400480,535,34
canal_prise,54.606107,12.164223,42.441885,19321,4
taux_absence_passe,45.676203,16.461884,29.214319,516,5
delai_jours,31.256307,6.122654,25.133653,7928,7
clinique,32.689295,9.431061,23.258234,1508,80
quartier,30.769231,9.349289,21.419941,502,30
cout_acte,23.191095,14.076782,9.114312,500,119
age,21.863799,14.369501,7.494298,501,78


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [148]:
# mesure 1 : la colonne piégée de la Q2 (nécessite colonne_piegee, ecart_non_filtre, ecart_filtre)
n_distinct = mon_train[colonne_piegee].nunique()

# mesure 2 : les lignes dupliquées, identifiant retiré
n_dup = int(train.drop(columns=[ID]).duplicated().sum())

# constat 1 : change une décision (écarter)
constat(f"la colonne {colonne_piegee} sépare tout et n'apprend rien",
        f"{n_distinct} valeurs distinctes sur {len(mon_train)} lignes, écart de {ecart_non_filtre:.1f} points",
        f"contre {ecart_filtre:.1f} points pour la meilleure colonne avec des groupes d'au moins 500 lignes",
        "écarter : chaque ligne forme son propre groupe, ce n'est pas un signal")

# constat 2 : négatif (défaut cherché)
constat("aucune ligne dupliquée trouvée" if n_dup == 0 else "des lignes dupliquées existent",
        f"{n_dup} ligne(s) dupliquée(s) sur {len(train)}",
        f"soit {100 * n_dup / len(train):.2f} % des lignes, contre 0 % attendu si la table était propre",
        "mesurer : si le nombre est nul on garde la table telle quelle, sinon on mesure ce que la suppression change")


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la colonne dossier_id sépare tout et n'apprend...,"149894 valeurs distinctes sur 149894 lignes, é...",contre 99.4 points pour la meilleure colonne a...,écarter : chaque ligne forme son propre groupe...


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la colonne dossier_id sépare tout et n'apprend...,"149894 valeurs distinctes sur 149894 lignes, é...",contre 99.4 points pour la meilleure colonne a...,écarter : chaque ligne forme son propre groupe...
4,aucune ligne dupliquée trouvée,0 ligne(s) dupliquée(s) sur 149894,"soit 0.00 % des lignes, contre 0 % attendu si ...",mesurer : si le nombre est nul on garde la tab...


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [149]:
# 1. âge impossible
print(train["age"].describe())
print("âges < 0 :", int((train["age"] < 0).sum()), "| âges > 120 :", int((train["age"] > 120).sum()))
print([c for c in train.columns if "age" in c.lower()])

# 2. nombre rangé en texte
for c in train.columns[~train.dtypes.map(pd.api.types.is_numeric_dtype)]:
    part = pd.to_numeric(train[c], errors="coerce").notna().mean()
    if part > 0.9:
        print(c, "ressemble à un nombre rangé en texte :", round(100 * part, 1), "%")

# 3. doublons (identifiant retiré)
print("doublons :", int(train.drop(columns=[ID]).duplicated().sum()))

count    149894.000000
mean         45.849560
std          18.792088
min          -1.000000
25%          33.000000
50%          46.000000
75%          58.000000
max          97.000000
Name: age, dtype: float64
âges < 0 : 46 | âges > 120 : 0
['age', 'age_mois']
doublons : 0


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [150]:
rng = np.random.default_rng(0)

pred_val = modele.predict(Xval)
bon = (pred_val == yval.to_numpy())   # un booléen par ligne de validation
N = len(bon)
p = bon.mean()

def tirages(n, k=20):
    """k scores (en %) mesurés sur n lignes tirées sans remise dans Xval."""
    return [100 * bon[rng.choice(N, size=n, replace=False)].mean() for _ in range(k)]

def formule(n):
    return 100 * (p * (1 - p) / n) ** 0.5

# 20 tirages de 1 000 lignes, puis de 5 000
ecarts_1000 = tirages(1000)
ecarts_5000 = tirages(5000)

# écart-type observé de chaque série de 20 scores
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

# barre annoncée par la formule du cours, et rapport observé / formule
formule_1000 = formule(1000)
formule_5000 = formule(5000)
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

print("N =", N)
print(ecart_type_1000, formule_1000, rapport_1000)
print(ecart_type_5000, formule_5000, rapport_5000)

N = 29979
1.251220456825631 1.201351670819712 1.041510564489324
0.40112473450484015 0.5372608001671654 0.7466108347752761


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [151]:
# 1. 20 tirages de 25 000 lignes, sans remise (même méthode que la Q5)
ecarts_25000 = tirages(25000)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))

# 2. barre de la formule, et rapport observé / formule
formule_25000 = formule(25000)
rapport_25000 = ecart_type_25000 / formule_25000

print("N =", N)
print(ecart_type_25000, formule_25000, rapport_25000)
print("rapports :", rapport_1000, rapport_5000, rapport_25000)

# 4. bonus (facultatif) : correction de population finie
correction = ((N - 25000) / (N - 1)) ** 0.5
print("correction :", correction, "| rapport observé :", rapport_25000)

N = 29979
0.10373326117347416 0.24027033416394242 0.43173561785906694
rapports : 1.041510564489324 0.7466108347752761 0.43173561785906694
correction : 0.4075395255361633 | rapport observé : 0.43173561785906694


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [152]:
reglages = []
for lr in [0.05, 0.5, 2]:
    m = HistGradientBoostingClassifier(learning_rate=lr, random_state=0).fit(Xtr, ytr)
    s = 100 * m.score(Xval, yval)
    reglages.append({"learning_rate": lr,
                     "score": s,
                     "arbres": int(m.n_iter_),
                     "en_barres": (s - reference) / erreur_type})

display(pd.DataFrame(reglages))

,learning_rate,score,arbres,en_barres
0,0.05,82.497748,71,3.412045
1,0.50,82.090797,15,1.557313
2,2.00,82.137496,10,1.770151


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [153]:
from sklearn.ensemble import RandomForestClassifier

# ── 1. TES PRÉDICTIONS (un signe et une raison, jamais une valeur) ──
prevoir("retirer zone_clinique", "...", "...")   # donnée
prevoir("max_leaf_nodes=15", "...", "...")        # réglage
prevoir("forêt aléatoire", "...", "...")          # famille de modèle

optimisme = "..."   # à toi, plus de 40 caractères : que fait au gain le choix du meilleur essai ?

# ── 2. LES MESURES ──
def decoupe(graine, table=None):
    return train_test_split(X if table is None else table, y,
                            test_size=0.2, stratify=y, random_state=graine)

def base(graine=0):
    a, b, c, d = decoupe(graine)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

def t_donnee(graine=0):
    a, b, c, d = decoupe(graine)
    a, b = a.drop(columns=["zone_clinique"]), b.drop(columns=["zone_clinique"])
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

def t_reglage(graine=0):
    a, b, c, d = decoupe(graine)
    return HistGradientBoostingClassifier(max_leaf_nodes=15, random_state=0).fit(a, c).score(b, d)

def t_famille(graine=0):
    a, b, c, d = decoupe(graine, X_num)
    return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0).fit(a, c).score(b, d)

essais = [("retirer zone_clinique", "donnée", t_donnee),
          ("max_leaf_nodes=15", "réglage", t_reglage),
          ("forêt aléatoire", "famille de modèle", t_famille)]

lignes = []
for nom, nature, f in essais:
    s = essai(nom, lambda f=f: f())          # la mesure est prise ici, après la prédiction
    if s is None:
        continue
    attendu = next(p["attendu"] for p in PREDICTIONS if p["essai"] == nom)
    gain = (s - score_interne) / erreur_type
    lignes.append({"essai": nom, "nature": nature, "attendu": attendu,
                   "mesuré": round(s, 2), "gain_en_barres": round(gain, 2),
                   "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"})

mes_essais = pd.DataFrame(lignes)
display(mes_essais)

# ── 3. LE SECOND TIRAGE du meilleur essai (autre graine) ──
meilleur = mes_essais.sort_values("gain_en_barres").iloc[-1]
f_meilleur = {nom: f for nom, _, f in essais}[meilleur["essai"]]
verifie_par_second_tirage = 100 * (f_meilleur(1) - base(1)) / erreur_type
print(meilleur["essai"], meilleur["gain_en_barres"], "→", verifie_par_second_tirage)

,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18
2,retirer zone_clinique,...,...,15:02:30


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18
2,retirer zone_clinique,...,...,15:02:30
3,max_leaf_nodes=15,...,...,15:02:30


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18
2,retirer zone_clinique,...,...,15:02:30
3,max_leaf_nodes=15,...,...,15:02:30
4,forêt aléatoire,...,...,15:02:30


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"
3,max_leaf_nodes=15,voir la question,...,82.53,"oui, à 15:02:30"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"
3,max_leaf_nodes=15,voir la question,...,82.53,"oui, à 15:02:30"
4,forêt aléatoire,voir la question,...,82.43,"oui, à 15:02:30"


,essai,nature,attendu,mesuré,gain_en_barres,verdict
0,retirer zone_clinique,donnée,...,82.51,0.00,dans le bruit
1,max_leaf_nodes=15,réglage,...,82.53,0.11,dans le bruit
2,forêt aléatoire,famille de modèle,...,82.43,-0.33,dans le bruit


max_leaf_nodes=15 0.11 → 0.045608157412282184


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [154]:
# 1. écart entre les deux familles, divisé par la barre de bruit
rapport = (score_interne - score_lineaire) / erreur_type
print(score_interne, score_lineaire, rapport)

# 2. coefficients comparables : les indicatrices sont toutes en 0/1, donc sur la même échelle
coef = pd.Series(modele_lineaire.coef_[0], index=Xtr_n.columns)
effectifs = Xtr_n.sum()

ind = [c for c in Xtr_n.columns
       if c.startswith(("quartier_", "assurance_")) and effectifs[c] >= 500]

tab = pd.DataFrame({"coef": coef[ind], "lignes": effectifs[ind]})
display(tab.reindex(tab["coef"].abs().sort_values(ascending=False).index).head(10))

trois_coefficients = list(tab["coef"].abs().sort_values(ascending=False).index[:3])
print(trois_coefficients)

82.50775542880015 82.2075452816972 1.368244722369708


,coef,lignes
quartier_Q15,-0.916162,1061.0
quartier_Q11,0.518558,1592.0
quartier_Q25,-0.515069,569.0
quartier_Q22,0.508102,563.0
quartier_Q19,0.486652,771.0
quartier_Q27,0.473567,507.0
quartier_Q02,-0.450570,16385.0
quartier_Q24,-0.442728,541.0
quartier_Q01,-0.344076,42897.0
quartier_Q07,0.336915,3186.0


['quartier_Q15', 'quartier_Q11', 'quartier_Q25']


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [155]:
# ── 1. LA FEATURE (remplace celle de la mise en place) ──
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    df["age_invalide"] = (df["age"] < 0).astype(int)
    return df

# ── 2. LE CONSTAT D'ORIGINE : doit être, mot pour mot, l'observation d'un constat de ta Q3 ──
constat("des âges impossibles dans la table",
        f"{int((train['age'] < 0).sum())} lignes avec un âge négatif",
        f"sur {len(train)} lignes, soit {100 * (train['age'] < 0).mean():.3f} %",
        "garder : on garde la colonne mais on signale les lignes concernées")
constat_d_origine = "des âges impossibles dans la table"

# ── 3. TA PRÉDICTION (signe et raison), AVANT la mesure ──
prevoir("ma feature age_invalide", "...", "...")

# ── 4. LA MESURE ──
score_avec_feature = essai("ma feature age_invalide", lambda: remesurer() / 100)
gain_de_ma_feature = (score_avec_feature - score_interne) / erreur_type
print("score avec feature :", score_avec_feature, "| gain en barres :", gain_de_ma_feature)

# ── 5. LE CONTOURNEMENT : table de validation préparée à la main, en oubliant .fillna("") ──
val_brut = train.loc[Xval.index]
d = val_brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in val_brut.columns]).copy()
for c in CAT:
    d[c] = d[c].astype("category").cat.set_categories(X[c].cat.categories)   # oubli : .fillna("")
d = ajouter_features(d)

score_contourne = 100 * (modele.predict(d) == yval).mean()
ecart_du_contournement = (score_contourne - score_avec_feature) / erreur_type
print("score contourné :", score_contourne, "| écart en barres :", ecart_du_contournement)

quelle_etape_oubliee = "..."   # à toi, plus de 20 caractères

,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la colonne dossier_id sépare tout et n'apprend...,"149894 valeurs distinctes sur 149894 lignes, é...",contre 99.4 points pour la meilleure colonne a...,écarter : chaque ligne forme son propre groupe...
4,aucune ligne dupliquée trouvée,0 ligne(s) dupliquée(s) sur 149894,"soit 0.00 % des lignes, contre 0 % attendu si ...",mesurer : si le nombre est nul on garde la tab...
5,des âges impossibles dans la table,46 lignes avec un âge négatif,"sur 149894 lignes, soit 0.031 %",garder : on garde la colonne mais on signale l...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18
2,retirer zone_clinique,...,...,15:02:30
3,max_leaf_nodes=15,...,...,15:02:30
4,forêt aléatoire,...,...,15:02:30
5,ma feature age_invalide,...,...,15:03:03


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"
3,max_leaf_nodes=15,voir la question,...,82.53,"oui, à 15:02:30"
4,forêt aléatoire,voir la question,...,82.43,"oui, à 15:02:30"
5,ma feature age_invalide,voir la question,...,82.51,"oui, à 15:03:03"


score avec feature : 82.50775542880015 | gain en barres : 0.0
score contourné : 82.50775542880015 | écart en barres : 0.0


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [156]:


# ── 1. écart de remplissage train / test, colonne par colonne (en points) ──
communes = [c for c in test.columns if c in train.columns]
rempl_train = 100 * train[communes].notna().mean()
rempl_test = 100 * test[communes].notna().mean()
ecarts = (pd.DataFrame({"train": rempl_train, "test": rempl_test,
                        "ecart": (rempl_train - rempl_test).abs()})
            .sort_values("ecart", ascending=False))
display(ecarts.head(6))

colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(ecarts.iloc[0]["ecart"])

# ── 2. part du test qui porte encore une valeur dans cette colonne ──
remplissage_du_test = float(rempl_test[colonne_trouvee])

# ── 3. gain interne, sur mon_train, même validation avec et sans la colonne ──
Xm, ym = X.loc[mon_train.index], y.loc[mon_train.index]
a, b, c, d = train_test_split(Xm, ym, test_size=0.2, stratify=ym, random_state=0)
barre = 100 * ((d.mean() * (1 - d.mean())) / len(d)) ** 0.5   # barre de VOTRE validation (plus petite)

col = train.loc[mon_train.index, colonne_trouvee]
if pd.api.types.is_numeric_dtype(col):
    plus = col
    trou = np.nan
else:
    cats = sorted(col.dropna().astype(str).unique()) + [""]
    plus = pd.Categorical(col.fillna("").astype(str), categories=cats)
    plus = pd.Series(plus, index=col.index)
    trou = ""

a_plus, b_plus = a.copy(), b.copy()
a_plus[colonne_trouvee] = plus.loc[a.index]
b_plus[colonne_trouvee] = plus.loc[b.index]

def fit(Xa):
    return HistGradientBoostingClassifier(random_state=0).fit(Xa, c)

s_sans = 100 * fit(a).score(b, d)
m_avec = fit(a_plus)
s_avec = 100 * m_avec.score(b_plus, d)

gain_interne = s_avec - s_sans
gain_en_barres = gain_interne / barre

# ── 4. simulation : la colonne devient manquante dans la validation, au taux de trous du test ──
rng = np.random.default_rng(0)
taux_trous = 1 - remplissage_du_test / 100
trouer = rng.random(len(b_plus)) < taux_trous

b_troue = b_plus.copy()
if trou == "":
    b_troue[colonne_trouvee] = pd.Categorical(
        np.where(trouer, "", b_plus[colonne_trouvee].astype(str)),
        categories=a_plus[colonne_trouvee].cat.categories)
else:
    b_troue.loc[trouer, colonne_trouvee] = np.nan

score_simule = 100 * m_avec.score(b_troue, d)
chute_en_barres = (s_avec - score_simule) / barre

print("colonne :", colonne_trouvee, "| écart :", ecart_remplissage, "| test renseigné :", remplissage_du_test)
print("sans :", s_sans, "| avec :", s_avec, "| gain :", gain_interne, "points,", gain_en_barres, "barres")
print("simulé :", score_simule, "| chute :", chute_en_barres, "barres | barre locale :", barre)

pourquoi_pas_la_derivee = "..."   # à toi, plus de 40 caractères
decision_A_ECARTER = "..."        # à toi, plus de 80 caractères


,train,test,ecart
canal_prise,100.000000,0.497481,99.502519
motif_absence,18.250897,19.003154,0.752257
rappel_envoye,33.734506,34.067262,0.332756
constantes_relevees,57.616716,57.346872,0.269844
solde_credit,94.502115,94.524568,0.022453
date_naissance,99.969312,99.965474,0.003837


colonne : canal_prise | écart : 99.50251879286263 | test renseigné : 0.49748120713736443
sans : 82.50775542880015 | avec : 85.47983588511958 | gain : 2.972080456319432 points, 13.322906556842579 barres
simulé : 82.46439174088529 | chute : 13.517292398861553 barres | barre locale : 0.22308048500069877


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [157]:
def suspects(df, cible, min_lignes=30, max_valeurs=50):
    """Rend une table, une ligne par colonne suspecte : colonne, écart de remplissage, signe, détail.
       Signe 1 (vide) : plus de 50 points d'écart de remplissage entre les deux groupes de la cible.
       Signe 2 (valeur) : colonne remplie des deux côtés, dont une valeur (d'au moins `min_lignes`
       lignes) ne vit que dans UN groupe. Ignore l'identifiant et la cible."""
    absent = df[cible] == 1                      # la cible ne sert qu'à séparer les deux groupes
    lignes = []
    for col in df.columns:
        if col in (cible, ID):
            continue
        r1 = 100 * df.loc[absent, col].notna().mean()
        r0 = 100 * df.loc[~absent, col].notna().mean()
        ecart = abs(r1 - r0)
        if ecart > 50:                           # signe 1
            lignes.append({"colonne": col, "ecart_remplissage": round(ecart, 1), "signe": "vide",
                           "detail": f"remplie à {r1:.1f} % chez les absents, {r0:.1f} % chez les présents"})
            continue
        if df[col].nunique() <= max_valeurs:     # signe 2 (on ignore les colonnes quasi uniques)
            ct = pd.crosstab(df[col], df[cible]).reindex(columns=[0, 1], fill_value=0)
            ct = ct[ct.sum(axis=1) >= min_lignes]
            seules = ct[(ct[0] == 0) | (ct[1] == 0)]
            if len(seules):
                ex = ", ".join(f"{v!r} ({'absents' if seules.loc[v, 1] > 0 else 'présents'} seulement, "
                               f"{int(seules.loc[v].sum())} lignes)" for v in seules.index[:3])
                lignes.append({"colonne": col, "ecart_remplissage": round(ecart, 1), "signe": "valeur",
                               "detail": ex})
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage", "signe", "detail"])

table_suspects = suspects(train, CIBLE)
display(table_suspects)

# ── le troisième signe : une valeur impossible sur peu de lignes ──
num = train.select_dtypes("number").drop(columns=[ID, CIBLE], errors="ignore")
scan = pd.DataFrame({"min": num.min(), "max": num.max(), "negatifs": (num < 0).sum(),
                     "lignes_au_min": (num == num.min()).sum()})
display(scan[scan["negatifs"] > 0])

porteuses = train["age"] == -1                   # à confirmer avec le scan ci-dessus
sentinelle_trouvee = "age = -1"
lignes_porteuses = int(porteuses.sum())
taux_sur_ces_lignes = 100 * train.loc[porteuses, CIBLE].mean()
print(lignes_porteuses, "lignes | taux :", taux_sur_ces_lignes, "% | taux global :", 100 * train[CIBLE].mean(), "%")

verdict_sentinelle = "..."   # à toi, plus de 40 caractères : fuite, ou défaut de saisie ?

,colonne,ecart_remplissage,signe,detail
0,motif_absence,100.0,vide,"remplie à 100.0 % chez les absents, 0.0 % chez..."


,min,max,negatifs,lignes_au_min
age,-1.0000,97.0000,46,46
duree_consultation,-15.0000,53.0000,27881,1
solde_credit,-19500.2500,12805.8600,12767,1
indice_risque_externe,-14.9300,121.0700,75,1
indice_meteo_jour,-33.0000,136.0000,383,1
age_mois,-12.0000,1164.0000,46,46
score_territorial,-4.5336,4.1072,74290,1


46 lignes | taux : 10.869565217391305 % | taux global : 18.250897300759203 %


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [158]:
# ── 1. TA PRÉDICTION, avant de regarder les tranches (plus de 60 caractères) ──
annonce = "..."   # le rappel aide-t-il ou nuit-il à délai égal ? et qu'est-ce qui te ferait changer d'avis ?

# ── 2. les colonnes (détectées par leur nom : à corriger si besoin) ──
COL_RAPPEL = next(c for c in mon_train.columns if "rappel" in c.lower())
COL_DELAI = next(c for c in mon_train.columns if "delai" in c.lower() or "délai" in c.lower())
print("rappel :", COL_RAPPEL, "| délai :", COL_DELAI)
print(mon_train[COL_RAPPEL].value_counts(dropna=False).head())
print(mon_train[COL_DELAI].describe())

avec = mon_train[COL_RAPPEL].astype(str).str.lower().isin(["1", "1.0", "true", "oui", "yes"])
delai = mon_train[COL_DELAI]
absent = mon_train[CIBLE]

# ── 3. l'écart global, en points ──
taux_avec, taux_sans = 100 * absent[avec].mean(), 100 * absent[~avec].mean()
ecart_global = taux_avec - taux_sans
print("taux avec rappel :", taux_avec, "| sans :", taux_sans, "| écart global :", ecart_global)

# ── 4. à délai égal : tranches de délai (à ajuster d'après le describe ci-dessus) ──
BORNES = [-np.inf, 0, 1, 3, 7, 14, 30, 60, np.inf]
tranche = pd.cut(delai, bins=BORNES, right=True)

tranches, tranches_exclues = [], []
for t in tranche.cat.categories:
    m = (tranche == t)
    n1, n0 = int((m & avec).sum()), int((m & ~avec).sum())
    if n1 == 0 or n0 == 0:
        tranches_exclues.append({"tranche": str(t), "lignes": n1 + n0, "raison": "un seul des deux groupes"})
        continue
    if n1 + n0 < 500:
        tranches_exclues.append({"tranche": str(t), "lignes": n1 + n0, "raison": "moins de 500 lignes"})
        continue
    p1, p0 = absent[m & avec].mean(), absent[m & ~avec].mean()
    barre_t = 100 * (p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0) ** 0.5   # barre d'un écart de deux taux
    ecart_t = 100 * (p1 - p0)
    tranches.append({"tranche": str(t), "lignes": n1 + n0, "n_avec": n1, "n_sans": n0,
                     "taux_avec": round(100 * p1, 2), "taux_sans": round(100 * p0, 2),
                     "ecart": round(ecart_t, 2), "barre": round(barre_t, 2),
                     "ecart_en_barres": round(ecart_t / barre_t, 2)})

display(pd.DataFrame(tranches))
print("tranches exclues :", tranches_exclues)

# ── 5. la variable qui explique l'écart global ──
delai_median_avec = float(delai[avec].median())
delai_median_sans = float(delai[~avec].median())
print("délai médian avec rappel :", delai_median_avec, "| sans :", delai_median_sans)

variable_qui_explique = "..."          # à toi, plus de 3 caractères : le nom de la variable, ou « aucune »
experience_qui_trancherait = "..."     # à toi, plus de 120 caractères : trois lignes

rappel : rappel_envoye | délai : delai_jours
rappel_envoye
NaN       99328
envoyé    50566
Name: count, dtype: int64
count    149894.000000
mean          6.062491
std          10.217476
min           0.000000
25%           0.000000
50%           1.000000
75%           5.000000
max          40.000000
Name: delai_jours, dtype: float64
taux avec rappel : nan | sans : 18.250897300759203 | écart global : nan


""


tranches exclues : [{'tranche': '(-inf, 0.0]', 'lignes': 59680, 'raison': 'un seul des deux groupes'}, {'tranche': '(0.0, 1.0]', 'lignes': 17301, 'raison': 'un seul des deux groupes'}, {'tranche': '(1.0, 3.0]', 'lignes': 18053, 'raison': 'un seul des deux groupes'}, {'tranche': '(3.0, 7.0]', 'lignes': 19412, 'raison': 'un seul des deux groupes'}, {'tranche': '(7.0, 14.0]', 'lignes': 14936, 'raison': 'un seul des deux groupes'}, {'tranche': '(14.0, 30.0]', 'lignes': 12584, 'raison': 'un seul des deux groupes'}, {'tranche': '(30.0, 60.0]', 'lignes': 7928, 'raison': 'un seul des deux groupes'}, {'tranche': '(60.0, inf]', 'lignes': 0, 'raison': 'un seul des deux groupes'}]
délai médian avec rappel : nan | sans : 1.0


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [159]:
COL_DELAI = next(c for c in train.columns if "delai" in c.lower() or "délai" in c.lower())

# ── 1. TROIS HYPOTHÈSES (signe attendu, test, et résultat précis qui la contredirait) ──
NOM = "delai_long (délai > 14 jours)"
hypothese(NOM,
          "un délai long entre la prise et le rendez-vous augmente l'absence (signe +)",
          "ajouter l'indicatrice et mesurer le gain en barres, sur la même validation que score_interne",
          "un gain inférieur à 2 barres en valeur absolue : la table n'apprend rien de plus avec cette indicatrice")
hypothese("age_senior (âge > 65 ans)",
          "les patients de plus de 65 ans sont moins souvent absents (signe −)",
          "comparer le taux d'absence des plus de 65 ans au taux global, avec la barre du groupe",
          "un écart de taux inférieur à 2 barres de bruit, ou un signe positif")
hypothese("montant_eleve (montant_acte > médiane)",
          "un acte coûteux est moins souvent manqué, car le patient s'y est engagé (signe −)",
          "comparer le taux d'absence au-dessus et en dessous de la médiane de montant_acte",
          "un écart de taux inférieur à 2 barres entre les deux moitiés")

# ── 2. LE TEST, sous le même nom que la première hypothèse ──
def avec_feature():
    Xc = X.copy()
    Xc["delai_long"] = (train[COL_DELAI] > 14).astype(int)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

def sans_feature():
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

prevoir(NOM, "plus", "...")        # à toi : le signe et la raison, AVANT la mesure
s = essai(NOM, avec_feature)
gain = (s - 100 * sans_feature()) / erreur_type
print("score :", s, "| gain en barres :", gain)

# ── 3. LE VERDICT, jugé sur le critère annoncé plus haut (|gain| ≥ 2 barres) ──
confirmee_ou_contredite = "confirmée" if gain >= 2 else "contredite"
print(confirmee_ou_contredite)

suite_donnee = "..."   # à toi, plus de 60 caractères : que fais-tu de ce résultat ?

,feature,ce qu'elle raconte,test,contredite si
0,delai_long (délai > 14 jours),un délai long entre la prise et le rendez-vous...,ajouter l'indicatrice et mesurer le gain en ba...,un gain inférieur à 2 barres en valeur absolue...


,feature,ce qu'elle raconte,test,contredite si
0,delai_long (délai > 14 jours),un délai long entre la prise et le rendez-vous...,ajouter l'indicatrice et mesurer le gain en ba...,un gain inférieur à 2 barres en valeur absolue...
1,age_senior (âge > 65 ans),les patients de plus de 65 ans sont moins souv...,comparer le taux d'absence des plus de 65 ans ...,un écart de taux inférieur à 2 barres de bruit...


,feature,ce qu'elle raconte,test,contredite si
0,delai_long (délai > 14 jours),un délai long entre la prise et le rendez-vous...,ajouter l'indicatrice et mesurer le gain en ba...,un gain inférieur à 2 barres en valeur absolue...
1,age_senior (âge > 65 ans),les patients de plus de 65 ans sont moins souv...,comparer le taux d'absence des plus de 65 ans ...,un écart de taux inférieur à 2 barres de bruit...
2,montant_eleve (montant_acte > médiane),"un acte coûteux est moins souvent manqué, car ...",comparer le taux d'absence au-dessus et en des...,un écart de taux inférieur à 2 barres entre le...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,15:02:18
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",15:02:18
2,retirer zone_clinique,...,...,15:02:30
3,max_leaf_nodes=15,...,...,15:02:30
4,forêt aléatoire,...,...,15:02:30
5,ma feature age_invalide,...,...,15:03:03
6,delai_long (délai > 14 jours),plus,...,15:03:10


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"
3,max_leaf_nodes=15,voir la question,...,82.53,"oui, à 15:02:30"
4,forêt aléatoire,voir la question,...,82.43,"oui, à 15:02:30"
5,ma feature age_invalide,voir la question,...,82.51,"oui, à 15:03:03"
6,delai_long (délai > 14 jours),voir la question,plus,82.51,"oui, à 15:03:10"


score : 82.50775542880015 | gain en barres : 0.0
contredite


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [160]:
COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

# on réutilise les mots de la liste : plus de risque d'apostrophe différente
IDENT, APRES, DISPA, UTIL = FAMILLES_ATTENDUES

familles = {
    "id_client":     IDENT,
    "date_defaut":   APRES,
    "statut_6_mois": APRES,
    "montant":       UTIL,
    "canal":         DISPA,
    "score_interne": UTIL,
}

justifications = {
    "id_client":     "Il désigne un client sans le décrire : il sépare chaque ligne des autres et n'apprend rien de général.",
    "date_defaut":   "Elle n'est renseignée que si l'emprunteur a fait défaut : elle contient la réponse, et n'existe pas au moment de décider.",
    "statut_6_mois": "Elle est relevée six mois après l'octroi : sa valeur n'existe pas encore quand on décide d'accorder le crédit.",
    "montant":       "Le montant demandé est connu au moment de la demande et décrit la ligne : on peut l'utiliser.",
    "canal":         "Elle est connue au moment de la demande, mais un canal peut changer ou ne plus être alimenté en production : à surveiller.",
    "score_interne": "Il est calculé au moment de la demande, donc disponible quand on décide : il reste utilisable.",
}

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))

  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [161]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la colonne dossier_id sépare tout et n'apprend...,"149894 valeurs distinctes sur 149894 lignes, é...",contre 99.4 points pour la meilleure colonne a...,écarter : chaque ligne forme son propre groupe...
4,aucune ligne dupliquée trouvée,0 ligne(s) dupliquée(s) sur 149894,"soit 0.00 % des lignes, contre 0 % attendu si ...",mesurer : si le nombre est nul on garde la tab...
5,des âges impossibles dans la table,46 lignes avec un âge négatif,"sur 149894 lignes, soit 0.031 %",garder : on garde la colonne mais on signale l...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,delai_long (délai > 14 jours),un délai long entre la prise et le rendez-vous...,ajouter l'indicatrice et mesurer le gain en ba...,un gain inférieur à 2 barres en valeur absolue...
1,age_senior (âge > 65 ans),les patients de plus de 65 ans sont moins souv...,comparer le taux d'absence des plus de 65 ans ...,un écart de taux inférieur à 2 barres de bruit...
2,montant_eleve (montant_acte > médiane),"un acte coûteux est moins souvent manqué, car ...",comparer le taux d'absence au-dessus et en des...,un écart de taux inférieur à 2 barres entre le...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 15:02:18"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 15:02:18"
2,retirer zone_clinique,voir la question,...,82.51,"oui, à 15:02:30"
3,max_leaf_nodes=15,voir la question,...,82.53,"oui, à 15:02:30"
4,forêt aléatoire,voir la question,...,82.43,"oui, à 15:02:30"
5,ma feature age_invalide,voir la question,...,82.51,"oui, à 15:03:03"
6,delai_long (délai > 14 jours),voir la question,plus,82.51,"oui, à 15:03:10"


VOS CONTRÔLES 



,contrôle,tient
0,les six colonnes sont classées,True
1,le vocabulaire est celui des quatre familles,True
2,la colonne du défaut n est pas classée « utili...,True
3,l identifiant est reconnu comme tel,True
4,le score calculé à la demande reste utilisable,True
5,"les six justifications sont écrites, une phras...",True


LA RELECTURE 

  À REVOIR  votre identifiant GitHub est renseigné   ->   les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant
  À REVOIR  cinq constats   ->   la question 3 en demande cinq — les trois de la séance plus deux des vôtres
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  ok        trois hypothèses
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  ok        une hypothèse a été testée sous son propre nom


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


In [162]:
ex = pd.read_csv(os.path.join(DOSSIER, "sample_submission.csv"))
sub = pd.read_csv("submission.csv")
print("même ordre que l'exemple :", (ex[ID].values == sub[ID].values).all())

p = pd.Series(predict(test).astype(int), index=test[ID].values)
final = ex[[ID]].copy()
final[CIBLE] = p.loc[final[ID]].values
final.to_csv("submission.csv", index=False)

chk = pd.read_csv("submission.csv")
print(chk.shape, (chk[ID].values == ex[ID].values).all(), chk[CIBLE].value_counts().to_dict())

même ordre que l'exemple : True
(63721, 2) True {0: 60229, 1: 3492}


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
